# Datathon Passos Mágicos — Limpeza da Base PEDE (2022, 2023, 2024)

Este notebook faz a **limpeza e padronização** da planilha `BASE_DE_DADOS_PEDE_2024_-_DATATHON.xlsx`,
que contém três abas (`PEDE2022`, `PEDE2023`, `PEDE2024`) com estrutura e nomenclatura de colunas
diferentes entre si.

**O que este notebook faz:**
1. Carrega as três abas
2. Remove colunas 100% vazias e colunas duplicadas
3. Corrige inconsistências de digitação/acentuação (`Gênero`, `Pedra`)
4. Corrige um bug de importação do Excel na aba 2023 (`Idade` / `Data de Nasc`)
5. Corrige a coluna `Fase` da aba 2024, que veio como código de turma em vez do número da fase
6. Padroniza nomes de colunas entre os três anos
7. Consolida tudo em uma tabela única em formato longo (1 linha = 1 aluno em 1 ano)
8. Valida os dados limpos (duplicidade de RA, faixa de valores dos indicadores)
9. Exporta os resultados (`.xlsx` e `.csv`)

> Ajuste o caminho do arquivo na célula abaixo (`RAW_PATH`) antes de rodar.

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 60)

RAW_PATH = "https://raw.githubusercontent.com/juliamchaves/dathatonfiap/99d60355d2eb2d39abd94b30e1ab9659b80b8c8a/data/raw/BASE%20DE%20DADOS%20PEDE%202024%20-%20DATATHON.xlsx"  # ajuste o caminho se necessário

## 1. Leitura dos dados brutos

In [ ]:
df_2022_raw = pd.read_excel(RAW_PATH, sheet_name="PEDE2022")
df_2023_raw = pd.read_excel(RAW_PATH, sheet_name="PEDE2023")
df_2024_raw = pd.read_excel(RAW_PATH, sheet_name="PEDE2024")

print("PEDE2022:", df_2022_raw.shape)
print("PEDE2023:", df_2023_raw.shape)
print("PEDE2024:", df_2024_raw.shape)

PEDE2022: (860, 42)
PEDE2023: (1014, 48)
PEDE2024: (1156, 50)


### Diagnóstico rápido dos problemas encontrados

Antes de limpar, vale registrar o que foi identificado ao inspecionar a base (isso justifica cada
etapa de limpeza feita mais abaixo):

- **Colunas 100% vazias** em 2023 e 2024 (`Cg`, `Cf`, `Ct`, `Rec Av1`...`Rec Av4`, `Indicado`,
  `Atingiu PV`, `Destaque IEG/IDA/IPV`, `Rec Psicologia`) — existem na planilha mas nunca foram
  preenchidas nesses anos.
- **Colunas duplicadas** (`Destaque IPV.1` em 2023, `Ativo/ Inativo.1` em 2024) com o mesmo
  conteúdo da coluna original.
- **`Gênero`** usa vocabulário diferente por ano: `Menina`/`Menino` (2022) vs. `Feminino`/`Masculino`
  (2023 e 2024).
- **`Pedra`** (categoria de desempenho) aparece grafada ora como `Ágata`, ora como `Agata`
  (sem acento) dentro da mesma coluna.
- **Bug de importação do Excel em 2023**: a coluna `Idade` tem parte dos valores convertidos para
  datas (`datetime(1900, 1, 8)` em vez do número `8`), e `Data de Nasc` mistura texto
  (`'6/17/2015'`) com objetos `datetime` já convertidos.
- **`Fase` em 2024** não contém o número da fase, e sim o código da turma (`'1A'`, `'8B'`, `'ALFA'`),
  diferente do que ocorre em 2022 (número puro) e 2023 (`'FASE 1'`...`'FASE 8'`, `'ALFA'`).
- **38 alunos em 2024** aparecem com `Fase = '9'` e `Pedra 2024 = 'INCLUIR'` — não é uma fase real,
  e sim um cadastro pendente de definição de turma.
- Nomes de colunas equivalentes variam entre anos (`Matem`/`Mat`, `Portug`/`Por`, `Inglês`/`Ing`,
  `INDE 22`/`INDE 2023`/`INDE 2024`, `Fase ideal`/`Fase Ideal`, `Defas`/`Defasagem` etc.).

## 2. Funções auxiliares de limpeza

In [ ]:
PEDRA_FIX = {"Agata": "Ágata", "AGATA": "Ágata", "agata": "Ágata"}


def drop_fully_empty_columns(df):
    """Remove colunas 100% nulas (campos legado sem preenchimento no ano)."""
    empty_cols = [c for c in df.columns if df[c].isna().all()]
    return df.drop(columns=empty_cols), empty_cols


def drop_duplicated_content_columns(df):
    """Remove colunas '.1' quando o conteúdo é idêntico ao da coluna original."""
    dupes = []
    for c in df.columns:
        if c.endswith(".1"):
            base = c[:-2]
            if base in df.columns and df[c].equals(df[base]):
                dupes.append(c)
    return df.drop(columns=dupes), dupes


def strip_strings(df):
    """Remove espaços extras/duplicados em todas as colunas de texto."""
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            df[c] = df[c].apply(
                lambda x: re.sub(r"\s+", " ", x).strip() if isinstance(x, str) else x
            )
    return df


def fix_accent_variants(series, mapping):
    return series.replace(mapping)

## 3. Limpeza — PEDE 2022

Colunas específicas: `Ano nasc`, `Idade 22`, `INDE 22`, `Pedra 22`, `Matem`, `Portug`, `Inglês`,
`Fase ideal`, `Defas`. `Fase` já vem como número (0 a 7).

In [ ]:
def clean_2022(df):
    df = df.copy()
    df, empty_cols = drop_fully_empty_columns(df)
    df = strip_strings(df)

    # Padroniza Gênero para o mesmo vocabulário de 2023/2024
    df["Gênero"] = df["Gênero"].replace({"Menina": "Feminino", "Menino": "Masculino"})

    # Corrige acentuação inconsistente em "Pedra"
    for c in ["Pedra 20", "Pedra 21", "Pedra 22"]:
        if c in df.columns:
            df[c] = fix_accent_variants(df[c], PEDRA_FIX)

    df = df.rename(columns={
        "Ano nasc": "Ano_Nascimento",
        "Idade 22": "Idade",
        "Instituição de ensino": "Instituicao_Ensino",
        "Pedra 22": "Pedra_Atual",
        "INDE 22": "INDE",
        "Matem": "Matematica",
        "Portug": "Portugues",
        "Inglês": "Ingles",
        "Fase ideal": "Fase_Ideal",
        "Defas": "Defasagem",
    })

    df["RA"] = df["RA"].astype(str).str.strip()
    df["Fase"] = pd.to_numeric(df["Fase"], errors="coerce")
    df["Fase_Num"] = df["Fase"]          # já é numérica em 2022
    df["Pendente_Inclusao"] = False       # não existe esse caso em 2022
    df["Ano"] = 2022
    return df, empty_cols


df_2022, dropped_2022 = clean_2022(df_2022_raw)
print("Colunas removidas em 2022:", dropped_2022)
print("Shape final 2022:", df_2022.shape)
df_2022.head(3)

Colunas removidas em 2022: []
Shape final 2022: (860, 45)


,RA,Fase,Turma,Nome,Ano_Nascimento,Idade,Gênero,Ano ingresso,Instituicao_Ensino,Pedra 20,Pedra 21,Pedra_Atual,INDE,Cg,Cf,Ct,Nº Av,Avaliador1,Rec Av1,Avaliador2,Rec Av2,Avaliador3,Rec Av3,Avaliador4,Rec Av4,IAA,IEG,IPS,Rec Psicologia,IDA,Matematica,Portugues,Ingles,Indicado,Atingiu PV,IPV,IAN,Fase_Ideal,Defasagem,Destaque IEG,Destaque IDA,Destaque IPV,Fase_Num,Pendente_Inclusao,Ano
0,RA-1,7,A,Aluno-1,2003,19,Feminino,2016,Escola Pública,Ametista,Ametista,Quartzo,5.783,753,18,10,4,Avaliador-5,Mantido na Fase atual,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Mantido na Fase atual,8.3,4.1,5.6,Requer avaliação,4.0,2.7,3.5,6.0,Sim,Não,7.278,5.0,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,7,False,2022
1,RA-2,7,A,Aluno-2,2005,17,Feminino,2017,Rede Decisão,Ametista,Ametista,Ametista,7.055,469,8,3,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase,Avaliador-31,Promovido de Fase + Bolsa,8.8,5.2,6.3,Sem limitações,6.8,6.3,4.5,9.7,Não,Não,6.778,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...,7,False,2022
2,RA-3,7,A,Aluno-3,2005,17,Feminino,2016,Rede Decisão,Ametista,Ametista,Ágata,6.591,629,13,6,4,Avaliador-5,Promovido de Fase,Avaliador-27,Promovido de Fase + Bolsa,Avaliador-28,Promovido de Fase + Bolsa,Avaliador-31,Promovido de Fase + Bolsa,0.0,7.9,5.6,Sem limitações,5.6,5.8,4.0,6.9,Não,Não,7.556,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...,7,False,2022


## 4. Limpeza — PEDE 2023

Além do padrão de renomeação, esta aba tem dois problemas específicos:

- **Colunas totalmente vazias / duplicadas** (ver diagnóstico acima)
- **Bug de importação do Excel**: valores pequenos de `Idade` viraram objetos `datetime`
  (`datetime(1900, 1, D)`, onde `D` é a idade real) porque a célula estava formatada como data.
  A correção extrai o dia (`.day`) desses casos e mantém os demais como número.
- **`Data de Nasc` mista** texto (`'6/17/2015'`) com `datetime` — padronizada via `pd.to_datetime`.
- **`Fase` em texto** (`'ALFA'`, `'FASE 1'`...`'FASE 8'`) — extraído o número em `Fase_Num`
  (`ALFA` → `0`).

In [ ]:
def clean_2023(df):
    df = df.copy()
    df, empty_cols = drop_fully_empty_columns(df)
    df, dup_cols = drop_duplicated_content_columns(df)
    df = strip_strings(df)

    # Corrige o bug de Idade importada como data (1900-01-DD)
    def fix_idade(v):
        if hasattr(v, "day") and hasattr(v, "month"):
            return v.day
        return v
    df["Idade"] = df["Idade"].apply(fix_idade)
    df["Idade"] = pd.to_numeric(df["Idade"], errors="coerce")

    # Padroniza Data de Nasc (mistura de string e datetime)
    df["Data de Nasc"] = pd.to_datetime(df["Data de Nasc"], errors="coerce")

    for c in ["Pedra 20", "Pedra 21", "Pedra 22", "Pedra 2023"]:
        if c in df.columns:
            df[c] = fix_accent_variants(df[c], PEDRA_FIX)

    df = df.rename(columns={
        "Nome Anonimizado": "Nome",
        "Data de Nasc": "Data_Nascimento",
        "Instituição de ensino": "Instituicao_Ensino",
        "Pedra 2023": "Pedra_Atual",
        "INDE 2023": "INDE",
        "Mat": "Matematica",
        "Por": "Portugues",
        "Ing": "Ingles",
        "Fase Ideal": "Fase_Ideal",
    })

    df["RA"] = df["RA"].astype(str).str.strip()

    def parse_fase(v):
        if not isinstance(v, str):
            return np.nan
        v = v.strip().upper()
        if v == "ALFA":
            return 0
        m = re.search(r"\d+", v)
        return int(m.group()) if m else np.nan

    df["Fase_Num"] = df["Fase"].apply(parse_fase)
    df["Pendente_Inclusao"] = False
    df["Ano"] = 2023
    return df, empty_cols + dup_cols


df_2023, dropped_2023 = clean_2023(df_2023_raw)
print("Colunas removidas em 2023:", dropped_2023)
print("Shape final 2023:", df_2023.shape)
print("Idade após correção (amostra):", sorted(df_2023["Idade"].dropna().unique())[:15])
df_2023.head(3)

Colunas removidas em 2023: ['Pedra 23', 'INDE 23', 'Cg', 'Cf', 'Ct', 'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 'Indicado', 'Atingiu PV', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Destaque IPV.1']
Shape final 2023: (1014, 35)
Idade após correção (amostra): [np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21)]


,RA,Fase,INDE,Pedra_Atual,Turma,Nome,Data_Nascimento,Idade,Gênero,Ano ingresso,Instituicao_Ensino,Pedra 20,Pedra 21,Pedra 22,INDE 22,Nº Av,Avaliador1,Avaliador2,Avaliador3,Avaliador4,IAA,IEG,IPS,IPP,IDA,Matematica,Portugues,Ingles,IPV,IAN,Fase_Ideal,Defasagem,Fase_Num,Pendente_Inclusao,Ano
0,RA-861,ALFA,9.31095,Topázio,ALFA A - G0/G1,Aluno-861,2015-06-17,8,Feminino,2023,Pública,NaN,NaN,NaN,NaN,2.0,Avaliador-11,Avaliador-2,NaN,NaN,9.5,10.0,8.13,8.4375,9.6,9.8,9.4,NaN,8.920,10.0,ALFA (1° e 2° ano),0,0,False,2023
1,RA-862,ALFA,8.22120,Topázio,ALFA A - G0/G1,Aluno-862,2014-05-31,9,Masculino,2023,Pública,NaN,NaN,NaN,NaN,2.0,Avaliador-11,Avaliador-2,NaN,NaN,8.5,9.1,8.14,7.5000,8.9,8.5,9.2,NaN,8.585,5.0,Fase 1 (3° e 4° ano),-1,0,False,2023
2,RA-863,ALFA,5.92975,Quartzo,ALFA A - G0/G1,Aluno-863,2016-02-25,7,Masculino,2023,Pública,NaN,NaN,NaN,NaN,2.0,Avaliador-11,Avaliador-2,NaN,NaN,0.0,7.6,3.14,5.9375,6.3,7.0,5.5,NaN,6.260,10.0,ALFA (1° e 2° ano),0,0,False,2023


## 5. Limpeza — PEDE 2024

Problema específico desta aba: a coluna `Fase` **não contém o número da fase**, e sim o
**código de turma** (`'1A'`, `'8B'`, `'ALFA'`...). O número da fase é extraído do primeiro dígito
do código (`ALFA` → `0`).

Também há 38 alunos com `Fase == '9'` e `Pedra 2024 == 'INCLUIR'`, que representam **cadastros
pendentes** (ainda sem turma/fase definida) e não uma fase real — eles são sinalizados na coluna
`Pendente_Inclusao` em vez de descartados, para não perder o registro do aluno.

In [ ]:
def clean_2024(df):
    df = df.copy()
    df, empty_cols = drop_fully_empty_columns(df)
    df, dup_cols = drop_duplicated_content_columns(df)
    df = strip_strings(df)

    df["Data de Nasc"] = pd.to_datetime(df["Data de Nasc"], errors="coerce")
    df["Idade"] = pd.to_numeric(df["Idade"], errors="coerce")

    for c in ["Pedra 20", "Pedra 21", "Pedra 22", "Pedra 23", "Pedra 2024"]:
        if c in df.columns:
            df[c] = fix_accent_variants(df[c], PEDRA_FIX)

    # Sinaliza alunos "a incluir" em vez de descartar o registro
    df["Pendente_Inclusao"] = df["Pedra 2024"].eq("INCLUIR")

    def parse_fase_2024(v):
        if not isinstance(v, str):
            return np.nan
        v = v.strip().upper()
        if v == "ALFA":
            return 0
        m = re.match(r"(\d+)", v)
        return int(m.group(1)) if m else np.nan

    df["Fase_Num"] = df["Fase"].astype(str).apply(parse_fase_2024)
    df.loc[df["Pendente_Inclusao"], "Fase_Num"] = np.nan

    df = df.rename(columns={
        "Nome Anonimizado": "Nome",
        "Data de Nasc": "Data_Nascimento",
        "Instituição de ensino": "Instituicao_Ensino",
        "Escola": "Nome_Escola",
        "Pedra 2024": "Pedra_Atual",
        "INDE 2024": "INDE",
        "Mat": "Matematica",
        "Por": "Portugues",
        "Ing": "Ingles",
        "Fase Ideal": "Fase_Ideal",
    })

    df["RA"] = df["RA"].astype(str).str.strip()
    df["INDE"] = pd.to_numeric(df["INDE"], errors="coerce")
    df["Ano"] = 2024
    return df, empty_cols + dup_cols


df_2024, dropped_2024 = clean_2024(df_2024_raw)
print("Colunas removidas em 2024:", dropped_2024)
print("Shape final 2024:", df_2024.shape)
print("Alunos pendentes de inclusão:", df_2024["Pendente_Inclusao"].sum())
df_2024.head(3)

Colunas removidas em 2024: ['Cg', 'Cf', 'Ct', 'Rec Av1', 'Rec Av2', 'Rec Psicologia', 'Indicado', 'Atingiu PV', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Ativo/ Inativo.1']
Shape final 2024: (1156, 41)
Alunos pendentes de inclusão: 38


,RA,Fase,INDE,Pedra_Atual,Turma,Nome,Data_Nascimento,Idade,Gênero,Ano ingresso,Instituicao_Ensino,Pedra 20,Pedra 21,Pedra 22,Pedra 23,INDE 22,INDE 23,Nº Av,Avaliador1,Avaliador2,Avaliador3,Avaliador4,Avaliador5,Avaliador6,IAA,IEG,IPS,IPP,IDA,Matematica,Portugues,Ingles,IPV,IAN,Fase_Ideal,Defasagem,Nome_Escola,Ativo/ Inativo,Pendente_Inclusao,Fase_Num,Ano
0,RA-1275,ALFA,7.611367,Ametista,ALFA A - G0/G1,Aluno-1275,2016-07-28,8,Masculino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,Avaliador-2,Avaliador-9,NaN,NaN,NaN,10.002,8.666667,6.26,5.625,8.0,10.0,6.0,NaN,5.446667,10.0,ALFA (1° e 2° ano),0,EE Chácara Florida II,Cursando,False,0.0,2024
1,RA-1276,ALFA,8.002867,Topázio,ALFA A - G0/G1,Aluno-1276,2016-10-16,8,Feminino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,Avaliador-2,Avaliador-9,NaN,NaN,NaN,10.002,9.333333,3.76,7.500,8.0,10.0,6.0,NaN,7.050000,10.0,ALFA (1° e 2° ano),0,EE Chácara Florida II,Cursando,False,0.0,2024
2,RA-1277,ALFA,7.952200,Ametista,ALFA A - G0/G1,Aluno-1277,2016-08-16,8,Masculino,2024,Pública,NaN,NaN,NaN,NaN,NaN,NaN,3,Avaliador-11,Avaliador-2,Avaliador-9,NaN,NaN,NaN,10.002,9.083333,3.76,7.500,8.0,10.0,6.0,NaN,7.046667,10.0,ALFA (1° e 2° ano),0,EE Dom Pedro Villas Boas de Souza,Cursando,False,0.0,2024


## 6. Validação dos dados limpos

In [ ]:
# RA não pode se repetir dentro do mesmo ano
for nome, df in [("2022", df_2022), ("2023", df_2023), ("2024", df_2024)]:
    dup = df["RA"].duplicated().sum()
    status = "OK" if dup == 0 else f"ALERTA: {dup} duplicados"
    print(f"RA duplicado em {nome}: {status}")

RA duplicado em 2022: OK
RA duplicado em 2023: OK
RA duplicado em 2024: OK


In [ ]:
# Indicadores (IAA, IEG, IPS, IPP, IDA, IPV, IAN, INDE) devem estar na faixa 0-10
INDICADORES = ["IAA", "IEG", "IPS", "IPP", "IDA", "IPV", "IAN", "INDE"]

for nome, df in [("2022", df_2022), ("2023", df_2023), ("2024", df_2024)]:
    for c in INDICADORES:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            fora = s[(s < 0) | (s > 10.5)]
            if len(fora):
                print(f"[ALERTA] {nome} / {c}: {len(fora)} valores fora da faixa esperada (0-10)")

print("Validação de faixa concluída (sem mensagens acima = tudo dentro do esperado).")

Validação de faixa concluída (sem mensagens acima = tudo dentro do esperado).


## 7. Consolidação em formato longo

Uma linha = **1 aluno em 1 ano**, com as colunas que existem de forma equivalente nos três anos.
Isso facilita comparações ao longo do tempo (2022 → 2023 → 2024), que é a base para responder às
perguntas de evolução de IAN, IDA, IEG etc. do desafio.

In [ ]:
COLUNAS_COMUNS = [
    "Ano", "RA", "Nome", "Fase", "Fase_Num", "Fase_Ideal", "Turma",
    "Idade", "Gênero", "Ano ingresso", "Instituicao_Ensino",
    "Pedra_Atual", "INDE", "IAA", "IEG", "IPS", "IPP", "IDA",
    "Matematica", "Portugues", "Ingles", "IPV", "IAN", "Defasagem",
    "Pendente_Inclusao",
]

def select_common(df):
    out = df[[c for c in COLUNAS_COMUNS if c in df.columns]].copy()
    for c in COLUNAS_COMUNS:
        if c not in out.columns:
            out[c] = np.nan
    return out[COLUNAS_COMUNS]

df_long = pd.concat(
    [select_common(df_2022), select_common(df_2023), select_common(df_2024)],
    ignore_index=True,
)

print("Shape do consolidado:", df_long.shape)
df_long.groupby("Ano")["RA"].nunique()

Shape do consolidado: (3030, 25)


,RA
Ano,
2022,860
2023,1014
2024,1156


In [ ]:
df_long.head(10)

## 8. Exportação

Gera:
- `PEDE_limpo.xlsx` — uma aba por ano já limpa, mais a aba `Consolidado_Longo`
- `PEDE_consolidado_longo.csv` — a versão consolidada em CSV, pronta para as análises e para o
  modelo preditivo

In [ ]:
with pd.ExcelWriter("PEDE_limpo.xlsx") as writer:
    df_2022.to_excel(writer, sheet_name="PEDE2022_limpo", index=False)
    df_2023.to_excel(writer, sheet_name="PEDE2023_limpo", index=False)
    df_2024.to_excel(writer, sheet_name="PEDE2024_limpo", index=False)
    df_long.to_excel(writer, sheet_name="Consolidado_Longo", index=False)

df_long.to_csv("PEDE_consolidado_longo.csv", index=False)

from google.colab import files

files.download("PEDE_limpo.xlsx")
files.download("PEDE_consolidado_longo.csv")

print("Arquivos exportados: PEDE_limpo.xlsx e PEDE_consolidado_longo.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Arquivos exportados: PEDE_limpo.xlsx e PEDE_consolidado_longo.csv
